# Evaluate DP Synthetic Data (post dp_wrapper.py fix)

Re-runs the same four evaluators over the **corrected** DP synthetic CSVs
for `diabetes_130us` and `acs_income` (9 combos each), generated after the
`dp_wrapper.py` fix (double batch_size division removed, WGAN-GP restored
for CTGAN/CopulaGAN, encoder+decoder jointly protected for TVAE).

The corrected CSVs live in a `new/` subfolder per dataset
(`data/synthetic/{dataset}/new/{label}.csv`) so the original (buggy) DP
CSVs are preserved untouched for comparison. `home_credit` stays in
`DATASETS` and is skipped via `FileNotFoundError` until its files land.

Output:
- Archives the previous `outputs/results/full_benchmark.csv` (pre-fix DP
  results) to `outputs/results/full_benchmark_pre_fix.csv` before overwriting
- Writes corrected `full_benchmark.csv`, `utility_scores.csv`, `privacy_scores.csv`
- Prints an old-vs-new comparison table so the fix's effect is directly visible


In [1]:
import os
os.chdir('..')

import json
import yaml
import pandas as pd
import numpy as np
from pathlib import Path

with open('config.yaml') as f:
    config = yaml.safe_load(f)

from src.evaluation.utility.statistical import StatisticalEvaluator
from src.evaluation.utility.tstr import TSTREvaluator
from src.evaluation.privacy.mia import MIAEvaluator
from src.evaluation.privacy.distance_metrics import DistanceMetricsEvaluator

stat_eval  = StatisticalEvaluator(config)
tstr_eval  = TSTREvaluator(config)
mia_eval   = MIAEvaluator(config)
dist_eval  = DistanceMetricsEvaluator(config)

# home_credit stays in the list on purpose — missing files are caught and
# logged per-combo below, so rerunning this notebook later (once the Kaggle
# run finishes) fills it in with zero code changes.
DATASETS   = ['diabetes_130us', 'acs_income', 'home_credit']
GENERATORS = config['generators']['list']                     # ['ctgan', 'tvae', 'copulagan']
EPSILONS   = config['differential_privacy']['epsilons']       # [1, 5, 10]

# Corrected DP CSVs (post dp_wrapper.py fix) live in a 'new/' subfolder per
# dataset, so the original buggy CSVs stay untouched alongside them for
# comparison. Datasets not listed here are read from the normal location
# (e.g. home_credit, until its corrected files exist).
SYNTH_SUBDIR_OVERRIDE = {
    'diabetes_130us': 'new',
    'acs_income':      'new',
}

print(f'Evaluators loaded. Epsilons from config: {EPSILONS}')
print(f'Reading corrected DP CSVs from: {SYNTH_SUBDIR_OVERRIDE}')

Evaluators loaded. Epsilons from config: [1, 5, 10]
Reading corrected DP CSVs from: {'diabetes_130us': 'new', 'acs_income': 'new'}


In [2]:
def load_dataset(config, dataset_name):
    """Load real train/test CSVs and restore categorical dtypes from meta.json."""
    processed_dir = Path(config['datasets'][dataset_name]['processed_dir'])
    train_df = pd.read_csv(processed_dir / 'train.csv')
    test_df  = pd.read_csv(processed_dir / 'test.csv')

    with open(processed_dir / 'meta.json') as f:
        meta = json.load(f)

    for col, col_meta in meta.get('columns', {}).items():
        if col_meta['type'] == 'categorical':
            if col in train_df.columns:
                train_df[col] = train_df[col].astype(str)
                test_df[col]  = test_df[col].astype(str)

    return train_df, test_df, meta


def load_synthetic(config, dataset_name, label, use_override=True):
    """
    Load a synthetic CSV. Raises FileNotFoundError if not yet generated.

    If use_override=True and dataset_name is in SYNTH_SUBDIR_OVERRIDE, reads
    from the 'new/' subfolder (corrected post-fix CSVs) instead of the base
    synthetic_dir. Set use_override=False to force reading the original
    (pre-fix) location, e.g. for the old-vs-new comparison below.
    """
    base_dir = Path(config['datasets'][dataset_name]['synthetic_dir'])
    if use_override and dataset_name in SYNTH_SUBDIR_OVERRIDE:
        base_dir = base_dir / SYNTH_SUBDIR_OVERRIDE[dataset_name]

    synth_path = base_dir / f'{label}.csv'
    if not synth_path.exists():
        raise FileNotFoundError(f'Synthetic file not found: {synth_path}')
    return pd.read_csv(synth_path)


def evaluate_combo(dataset_name, target_col, real_train, real_test, gen_name, epsilon_label, label, use_override=True):
    """Run all 4 evaluators for one (dataset, generator, epsilon) combo and return a flat row dict."""
    synthetic = load_synthetic(config, dataset_name, label, use_override=use_override)

    # Align synthetic dtypes with real (same pattern as baseline notebook)
    for col in synthetic.columns:
        if col in real_train.columns:
            if real_train[col].dtype == object:
                synthetic[col] = synthetic[col].astype(str)

    stat_res = stat_eval.evaluate(real_train, synthetic, dataset_name, label)
    tstr_res = tstr_eval.evaluate(real_train, real_test, synthetic, target_col, dataset_name, label)
    mia_res  = mia_eval.evaluate(real_train, real_test, synthetic, target_col, dataset_name, label)
    dist_res = dist_eval.evaluate(real_train, real_test, synthetic, target_col, dataset_name, label)

    return {
        'dataset':            dataset_name,
        'generator':          gen_name,
        'epsilon':            epsilon_label,
        'label':              label,
        # Utility
        'stat_overall':       stat_res['overall_score'],
        'wasserstein_mean':   stat_res['wasserstein_mean'],
        'corr_diff':          stat_res['correlation_matrix_diff'],
        'cat_similarity':     stat_res['categorical_similarity_mean'],
        'tstr_auc':           tstr_res['tstr']['roc_auc'],
        'trtr_auc':           tstr_res['trtr']['roc_auc'],
        'tstr_f1':            tstr_res['tstr']['f1_weighted'],
        'tstr_accuracy':      tstr_res['tstr']['accuracy'],
        'utility_ratio':      tstr_res['utility_ratio'],
        # Privacy
        'mia_auc':            mia_res['mia_auc'],
        'mia_privacy_score':  mia_res['privacy_score'],
        'dcr_mean':           dist_res['dcr_mean'],
        'dcr_median':         dist_res['dcr_median'],
        'nndr_mean':          dist_res['nndr_mean'],
        'nndr_median':        dist_res['nndr_median'],
        'dist_privacy_score': dist_res['privacy_score'],
    }


print('Helper functions defined (load_synthetic now respects SYNTH_SUBDIR_OVERRIDE).')

Helper functions defined (load_synthetic now respects SYNTH_SUBDIR_OVERRIDE).


In [3]:
dp_results = []
skipped = []

for dataset_name in DATASETS:
    print(f'\n{"="*60}')
    print(f'  Dataset: {dataset_name}')
    print(f'{"="*60}')

    target_col = config['datasets'][dataset_name]['target_col']

    try:
        real_train, real_test, meta = load_dataset(config, dataset_name)
    except FileNotFoundError as e:
        print(f'  SKIPPING dataset entirely — processed data not found: {e}')
        skipped.append((dataset_name, 'ALL', 'ALL'))
        continue

    for gen_name in GENERATORS:
        for eps in EPSILONS:
            label = f'{gen_name}_eps{eps}'
            print(f'\n  [{label}]')

            try:
                row = evaluate_combo(dataset_name, target_col, real_train, real_test, gen_name, eps, label)
                dp_results.append(row)
                print(f'    TSTR AUC: {row["tstr_auc"]:.4f} | MIA AUC: {row["mia_auc"]:.4f} | DCR: {row["dcr_mean"]:.4f}')

            except FileNotFoundError as e:
                print(f'    SKIPPED (not generated yet): {e}')
                skipped.append((dataset_name, label, str(e)))

            except Exception as e:
                print(f'    ERROR: {e}')
                skipped.append((dataset_name, label, f'ERROR: {e}'))

print(f'\n\nDP evaluation complete. {len(dp_results)} combos scored, {len(skipped)} skipped.')
if skipped:
    print('\nSkipped combos (expected for home_credit until Kaggle run finishes):')
    for s in skipped:
        print(f'  {s[0]} / {s[1]}')


  Dataset: diabetes_130us

  [ctgan_eps1]
    TSTR AUC: 0.6189 | MIA AUC: 0.6021 | DCR: 7.5445

  [ctgan_eps5]
    TSTR AUC: 0.6044 | MIA AUC: 0.6163 | DCR: 7.4416

  [ctgan_eps10]
    TSTR AUC: 0.6002 | MIA AUC: 0.6026 | DCR: 7.5986

  [tvae_eps1]
    TSTR AUC: 0.4998 | MIA AUC: 0.6989 | DCR: 5.6441

  [tvae_eps5]
    TSTR AUC: 0.5223 | MIA AUC: 0.6899 | DCR: 6.7274

  [tvae_eps10]
    TSTR AUC: 0.5151 | MIA AUC: 0.6982 | DCR: 6.7597

  [copulagan_eps1]
    TSTR AUC: 0.5546 | MIA AUC: 0.6796 | DCR: 46.1515

  [copulagan_eps5]
    TSTR AUC: 0.5713 | MIA AUC: 0.6695 | DCR: 46.5773

  [copulagan_eps10]
    TSTR AUC: 0.5776 | MIA AUC: 0.6856 | DCR: 47.3850

  Dataset: acs_income

  [ctgan_eps1]
    TSTR AUC: 0.8771 | MIA AUC: 0.6106 | DCR: 25.3019

  [ctgan_eps5]
    TSTR AUC: 0.8769 | MIA AUC: 0.6167 | DCR: 25.9816

  [ctgan_eps10]
    TSTR AUC: 0.8773 | MIA AUC: 0.6211 | DCR: 27.4121

  [tvae_eps1]
    TSTR AUC: 0.6024 | MIA AUC: 0.8930 | DCR: 8.9950

  [tvae_eps5]
    TSTR AUC: 0.4183

## Archive pre-fix results, merge, and write Stage 6-ready outputs

- Archives the existing `full_benchmark.csv` (pre-fix DP rows) to
  `outputs/results/full_benchmark_pre_fix.csv`, if not already archived —
  keeps the buggy run's numbers on record for a before/after comparison.
- `full_benchmark.csv`: baseline (nodp) + corrected DP rows, wide format.
- `utility_scores.csv` / `privacy_scores.csv`: same rows split per
  `config['reporting']`.

In [4]:
dp_df = pd.DataFrame(dp_results)

reporting_cfg = config['reporting']
full_path = Path(reporting_cfg['output_file'])

# Archive the pre-fix full_benchmark.csv (contains the old, buggy DP rows)
# before it gets overwritten — keep it on record for the before/after
# comparison in the next cell, and for the report appendix.
archive_path = full_path.parent / 'full_benchmark_pre_fix.csv'
if full_path.exists() and not archive_path.exists():
    import shutil
    shutil.copy(full_path, archive_path)
    print(f'Archived pre-fix results to {archive_path}')
elif archive_path.exists():
    print(f'Pre-fix archive already exists at {archive_path} — not overwriting.')
else:
    print('No existing full_benchmark.csv found to archive (first run?).')

baseline_path = Path('outputs/results/baseline_scores.csv')
if baseline_path.exists():
    baseline_df = pd.read_csv(baseline_path)
    full_df = pd.concat([baseline_df, dp_df], ignore_index=True)
    print(f'Merged {len(baseline_df)} baseline rows + {len(dp_df)} corrected DP rows = {len(full_df)} total rows.')
else:
    full_df = dp_df
    print(f'WARNING: {baseline_path} not found — writing DP-only rows ({len(dp_df)}).')

# Dedup safety net: if this notebook is rerun (e.g. once home_credit lands),
# keep the latest row per (dataset, label) instead of duplicating.
full_df = full_df.drop_duplicates(subset=['dataset', 'label'], keep='last').reset_index(drop=True)

utility_cols = ['dataset', 'generator', 'epsilon', 'label',
                 'stat_overall', 'wasserstein_mean', 'corr_diff', 'cat_similarity',
                 'tstr_auc', 'trtr_auc', 'tstr_f1', 'tstr_accuracy', 'utility_ratio']
privacy_cols = ['dataset', 'generator', 'epsilon', 'label',
                 'mia_auc', 'mia_privacy_score',
                 'dcr_mean', 'dcr_median', 'nndr_mean', 'nndr_median', 'dist_privacy_score']

full_path.parent.mkdir(parents=True, exist_ok=True)

full_df.to_csv(reporting_cfg['output_file'], index=False)
full_df[utility_cols].to_csv(reporting_cfg['utility_file'], index=False)
full_df[privacy_cols].to_csv(reporting_cfg['privacy_file'], index=False)

print(f"\nSaved (corrected):")
print(f"  {reporting_cfg['output_file']}  ({len(full_df)} rows)")
print(f"  {reporting_cfg['utility_file']}")
print(f"  {reporting_cfg['privacy_file']}")

print(f"\nRows per dataset:")
print(full_df.groupby('dataset').size())
print(f"\nRows per (dataset, epsilon):")
print(full_df.groupby(['dataset', 'epsilon']).size())

Archived pre-fix results to outputs/results/full_benchmark_pre_fix.csv
Merged 9 baseline rows + 18 corrected DP rows = 27 total rows.

Saved (corrected):
  outputs/results/full_benchmark.csv  (27 rows)
  outputs/results/utility_scores.csv
  outputs/results/privacy_scores.csv

Rows per dataset:
dataset
acs_income        12
diabetes_130us    12
home_credit        3
dtype: int64

Rows per (dataset, epsilon):
dataset         epsilon
acs_income      1          3
                5          3
                10         3
                nodp       3
diabetes_130us  1          3
                5          3
                10         3
                nodp       3
home_credit     nodp       3
dtype: int64


## Before / after: did the dp_wrapper.py fix actually work?

Compares each corrected DP row against its pre-fix counterpart
(archived above) on the two metrics that mattered most in the
diagnosis: `utility_ratio` and `mia_auc`. Positive `mia_auc_delta`
means MIA AUC dropped (moved toward 0.5, i.e. *more* private) after
the fix — that's the direction we expect but did not see pre-fix.

In [5]:
archive_path = Path(reporting_cfg['output_file']).parent / 'full_benchmark_pre_fix.csv'

if archive_path.exists():
    pre_df = pd.read_csv(archive_path)
    pre_dp = pre_df[pre_df['epsilon'].astype(str).isin(['1', '5', '10'])]
    post_dp = full_df[full_df['epsilon'].astype(str).isin(['1', '5', '10'])]

    merged = pre_dp.merge(
        post_dp, on=['dataset', 'generator', 'epsilon'],
        suffixes=('_old', '_new'),
    )
    merged['utility_ratio_delta'] = merged['utility_ratio_new'] - merged['utility_ratio_old']
    merged['mia_auc_delta'] = merged['mia_auc_old'] - merged['mia_auc_new']  # positive = improved privacy

    cols = ['dataset', 'generator', 'epsilon',
            'utility_ratio_old', 'utility_ratio_new', 'utility_ratio_delta',
            'mia_auc_old', 'mia_auc_new', 'mia_auc_delta']
    comparison = merged[cols].sort_values(['dataset', 'generator', 'epsilon'])
    print(f'{len(comparison)} combos compared (only datasets/generators present in both runs).')
    comparison
else:
    print(f'No pre-fix archive found at {archive_path} — nothing to compare (first run on this data).')
    comparison = None

comparison

0 combos compared (only datasets/generators present in both runs).


,dataset,generator,epsilon,utility_ratio_old,utility_ratio_new,utility_ratio_delta,mia_auc_old,mia_auc_new,mia_auc_delta


In [6]:
# Quick sanity check: utility should trend down and MIA AUC should trend
# toward 0.5 as epsilon decreases (Stage 5 checkpoint from project_context.md).
# 'nodp' sorts after numeric epsilons here, which is fine for eyeballing.
pivot = full_df.pivot_table(
    index=['dataset', 'generator'],
    columns='epsilon',
    values=['utility_ratio', 'mia_auc'],
)
pivot

mia_auc                                \
epsilon                          1         5        10      nodp   
dataset        generator                                           
acs_income     copulagan  0.692379  0.680341  0.704911  0.604143   
               ctgan      0.610569  0.616746  0.621084  0.637356   
               tvae       0.893039  0.827285  0.803298  0.689626   
diabetes_130us copulagan  0.679559  0.669453  0.685558  0.615658   
               ctgan      0.602126  0.616256  0.602615  0.636841   
               tvae       0.698874  0.689899  0.698155  0.664701   
home_credit    copulagan       NaN       NaN       NaN  0.695781   
               ctgan           NaN       NaN       NaN  0.689894   
               tvae            NaN       NaN       NaN  0.689539   

                         utility_ratio                                
epsilon                              1         5        10      nodp  
dataset        generator                                              
acs_income     copulagan      0.922375  0.925412  0.928303  0.972927  
               ctgan          0.979629  0.979349  0.979872  0.974668  
               tvae           0.674655  0.471973  0.513905  0.961008  
diabetes_130us copulagan      0.811760  0.836254  0.845454  0.805716  
               ctgan          0.905906  0.884728  0.878566  0.843707  
               tvae           0.732871  0.752510  0.740995  0.903520  
home_credit    copulagan           NaN       NaN       NaN  0.921013  
               ctgan               NaN       NaN       NaN  0.922612  
               tvae                NaN       NaN       NaN  0.817494